# 05 · CADS Simulation Results — Phase 5

**Prerequisites:** `python -m src.simulation.sweep` (300 CADS + 12 baseline runs, ~20–30 min)

⚠️ **Carbon profiles are synthetic** (diurnal sinusoidal model). Rerun after Electricity Maps data arrives.

⚠️ **Start times are estimated** from `schedule_delay_sec / 3600`. Jobs with `schedule_delay_sec=0` are assigned `rel_hour=0` and flagged `start_uncertain=True`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from src.config import FIG_DIR, TABLE_DIR, INTERIM_DIR
from src.viz.style import set_ieee_style

set_ieee_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SWEEP = Path('outputs/sim_runs/sweep_results.parquet')
if not SWEEP.exists():
    print('Run `python -m src.simulation.sweep` first.')
else:
    df   = pd.read_parquet(SWEEP)
    cads = df[df['run_type'] == 'cads'].copy()
    base = df[df['run_type'] == 'baseline'].copy()
    print(f'Loaded: {len(cads)} CADS runs, {len(base)} baseline runs')
    print(cads[['region','threshold','max_defer','reduction_pct']].describe())

## Cell 1 — Headline: best CADS config per region

In [ ]:
best_per_region = (
    cads.groupby(['region', 'threshold', 'max_defer'])
    ['reduction_pct'].mean()
    .reset_index()
    .sort_values('reduction_pct', ascending=False)
    .groupby('region').first()
    .reset_index()
)

print('Best CADS configuration per region (anchor-averaged):')
print(best_per_region.to_string(index=False))

low      = best_per_region['reduction_pct'].min()
high     = best_per_region['reduction_pct'].max()
best_p95 = cads.loc[cads['reduction_pct'].idxmax(), 'p95_defer_hours']

print(f'\nHEADLINE: Under best config, CADS reduces GPU workload carbon by '
      f'{low:.1f}\u2013{high:.1f}% across regions and anchors, '
      f'with p95 deferral of {best_p95:.1f} hours')

## Cell 2 — Figure 8: CO\u2082 reduction vs max_defer_hours

In [ ]:
regions    = sorted(cads['region'].unique())
thresholds = sorted(cads['threshold'].unique())
colors     = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#D55E00']

fig, axes = plt.subplots(1, len(regions), figsize=(14, 3.2), sharey=True)

for ax, region in zip(axes, regions):
    sub = cads[cads['region'] == region]
    for i, thr in enumerate(thresholds):
        tsub = sub[sub['threshold'] == thr]
        grp  = tsub.groupby('max_defer')['reduction_pct']
        mn, lo, hi = grp.mean(), grp.min(), grp.max()
        ax.plot(mn.index, mn.values, color=colors[i], marker='o',
                ms=4, label=f'{int(thr)} g')
        ax.fill_between(mn.index, lo.values, hi.values,
                        alpha=0.15, color=colors[i])
    ax.set_title(region, fontsize=8)
    ax.set_xlabel('Max defer (h)')
    ax.set_xticks([1, 2, 4, 8, 12])

axes[0].set_ylabel('CO\u2082 reduction (%)')
axes[-1].legend(title='Threshold', fontsize=6)
fig.suptitle('Figure 8 \u2014 CO\u2082 reduction vs deferral window '
             '(bands = min/max across anchors)', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_reduction_vs_defer.pdf')
plt.show()
print('Saved fig8_reduction_vs_defer.pdf')

## Cell 3 — Figure 9: Pareto frontier (delay cost vs carbon benefit)

In [ ]:
anchors       = sorted(cads['anchor'].unique())
region_colors = dict(zip(regions, ['#0072B2', '#E69F00', '#009E73', '#CC79A7']))

fig, axes = plt.subplots(1, len(anchors), figsize=(12, 3.5), sharey=True)

for ax, anchor in zip(axes, anchors):
    sub = cads[cads['anchor'] == anchor]
    for region in regions:
        rsub = sub[sub['region'] == region]
        ax.scatter(rsub['p95_defer_hours'], rsub['reduction_pct'],
                   color=region_colors[region], s=18, alpha=0.7, label=region)
    ax.set_title(anchor, fontsize=8)
    ax.set_xlabel('p95 deferral (h)')

axes[0].set_ylabel('CO\u2082 reduction (%)')
hl, ll = axes[0].get_legend_handles_labels()
axes[-1].legend(dict(zip(ll, hl)).values(), dict(zip(ll, hl)).keys(), fontsize=6)
fig.suptitle('Figure 9 \u2014 Pareto: job delay vs carbon saving '
             '(each point = one (threshold, max_defer) combo)', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_pareto.pdf')
plt.show()
print('Saved fig9_pareto.pdf')

## Cell 4 — Figure 10: Deferral eligibility breakdown by GPU-hours

In [ ]:
cols = ['job_type_public', 'gpu_request', 'duration_hours',
        'is_deferrable', 'priority_class',
        'left_truncated', 'right_truncated']
jobs = pd.read_parquet(INTERIM_DIR / 'job_summary_canonical.parquet', columns=cols)
jobs = jobs[~jobs['left_truncated'] & ~jobs['right_truncated']]
jobs['gpu_hours'] = jobs['gpu_request'] * jobs['duration_hours']

def _cat(row):
    if row['is_deferrable']:                             return 'Deferrable'
    if str(row.get('priority_class','')).lower() in ('high','critical'): return 'Non-defer HP'
    return 'Non-defer LP'

jobs['cat'] = jobs.apply(_cat, axis=1)
bd  = jobs.groupby('cat')['gpu_hours'].sum()
pct = bd / bd.sum() * 100

fig, ax = plt.subplots(figsize=(5, 2.5))
bar_colors = {'Deferrable': '#009E73', 'Non-defer HP': '#D55E00', 'Non-defer LP': '#0072B2'}
bottom = 0
for cat, val in pct.items():
    ax.barh('GPU-hours', val, left=bottom, color=bar_colors.get(cat,'#999'),
            label=f'{cat} ({val:.1f}%)')
    if val > 4:
        ax.text(bottom + val/2, 0, f'{val:.1f}%', ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
    bottom += val
ax.set_xlim(0, 100)
ax.set_xlabel('% of total GPU-hours')
ax.set_title('Figure 10 \u2014 Deferral eligibility by GPU-hours')
ax.legend(fontsize=6, loc='upper right', bbox_to_anchor=(1.0, -0.25), ncol=3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_eligibility.pdf')
plt.show()
print(pct.to_string())

## Cell 5 — Table 2 (paper-ready): best config per region

In [ ]:
table2 = (
    cads.groupby(['region', 'threshold', 'max_defer'])
    .agg(reduction_pct=('reduction_pct','mean'),
         p95_defer_hours=('p95_defer_hours','mean'),
         jobs_deferred_pct=('jobs_deferred_pct','mean'))
    .reset_index()
    .sort_values('reduction_pct', ascending=False)
    .groupby('region').first().reset_index()
    [['region','threshold','max_defer',
      'reduction_pct','p95_defer_hours','jobs_deferred_pct']]
)
table2.columns = ['Region','Threshold (gCO\u2082/kWh)','Max defer (h)',
                  'Reduction (%)','p95 delay (h)','Jobs deferred (%)']
styled = (table2.style
    .format({'Reduction (%)':'{:.2f}','p95 delay (h)':'{:.1f}','Jobs deferred (%)':'{:.2f}'})
    .set_caption('Table 2 \u2014 Best CADS config per region (anchor-averaged)'))
display(styled)
table2.to_csv(TABLE_DIR / 'table2_best_cads_per_region.csv', index=False)
print('Saved table2_best_cads_per_region.csv')

## Cell 6 — Write phase5_headline.json

In [ ]:
cols2 = ['gpu_request','duration_hours','is_deferrable',
         'left_truncated','right_truncated']
j2 = pd.read_parquet(INTERIM_DIR / 'job_summary_canonical.parquet', columns=cols2)
j2 = j2[~j2['left_truncated'] & ~j2['right_truncated']]
j2['gpu_hours'] = j2['gpu_request'] * j2['duration_hours']

def_frac_jobs = float(j2['is_deferrable'].mean())
def_frac_gh   = float(j2.loc[j2['is_deferrable'],'gpu_hours'].sum() / j2['gpu_hours'].sum())

headline = {
    'best_reduction_pct_range':         [round(float(low),2), round(float(high),2)],
    'best_p95_defer_hours':              round(float(best_p95), 2),
    'deferrable_fraction_of_jobs':       round(def_frac_jobs, 4),
    'deferrable_fraction_of_gpu_hours':  round(def_frac_gh, 4),
    'note': ('synthetic carbon profiles \u2014 pending real Electricity Maps data; '
             'start times estimated from schedule_delay_sec'),
}

out_json = TABLE_DIR / 'phase5_headline.json'
with open(out_json, 'w') as f:
    json.dump(headline, f, indent=2)
print('=== Phase 5 Headline ===')
print(json.dumps(headline, indent=2))
print(f'Saved {out_json}')